# Cointegration Validation (Out-of-Sample)

This notebook tests whether the three cointegrated pairs identified in `01_spread_exploration.ipynb` --- GS/MS, KO/PEP, and DAL/UAL --- remain cointegrated over the out-of-sample period (January 2024 -- December 2025). Spreads are reconstructed using in-sample hedge ratios and intercepts; stationarity is re-assessed via ADF and KPSS tests. Structural stability is evaluated with a Chow test, and a fresh Engle--Granger screen is run on all 16 pairs using OOS prices only.

**Pipeline overview.**

| Step | Function | Description |
|------|----------|-------------|
| 1 | Load IS parameters | Reload cointegrated pairs, hedge ratios, and log prices from `data/processed/` |
| 2 | Load OOS prices | Retrieve daily closing prices for January 2024 -- December 2025 |
| 3 | Reconstruct OOS spreads | Apply IS hedge ratios to OOS log prices; compare IS vs OOS spread statistics |
| 4 | ADF + KPSS tests | Test stationarity of OOS spreads under IS parameters |
| 5 | Chow structural break test | Test for a parameter shift at the IS/OOS boundary |
| 6 | OOS Engle--Granger re-screen | Re-run cointegration tests on all 16 pairs using OOS prices only |
| 7 | Visualise | Concatenated spread levels, z-scores, and rolling half-life across both periods |

**Structural break test.** The Chow $F$-test evaluates whether the hedge ratio and intercept of the cointegrating regression remain stable at the IS/OOS boundary. It compares the residual sum of squares from a pooled regression against separate regressions on each sub-period. Rejection indicates a regime change, invalidating the use of IS parameters for OOS signal generation.

**Key finding.** All three pairs exhibit structural breaks (Chow $F > 269$, $p < 0.001$) and fail the ADF stationarity test under IS parameters out-of-sample ($p > 0.45$ for all pairs). Only GS/MS retains a marginal OOS cointegration signal when re-estimated on OOS prices alone ($p = 0.019$); KO/PEP and DAL/UAL degrade entirely.

**Outputs.** OOS prices and validation statistics are saved to `data/processed/` for use in `04_backtest_results.ipynb`.

## Imports

In [1]:
import sys
import os
os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore, compute_rolling_half_life # spread analysis

## Load In-Sample Spread Parameters

Reload cointegrated pairs (with hedge ratios and intercepts), spread summary statistics, and log prices saved by `01_spread_exploration.ipynb`.

In [2]:
# load the cointegrated pairs and their stats (hedge ratio, intercept, adf stat, p-value)
cointegrated_pairs = pd.read_csv("../../data/processed/cointegrated_pairs.csv")

# load their summary of spread data (half-life, hurst exponent, z-score, adf-stat, p-value, spread mean, spread std)
summary_df = pd.read_csv("../../data/processed/spread_summary.csv", index_col = 0)

# load the prices log
prices_log = pd.read_csv("../../data/processed/prices_log.csv", index_col=0, parse_dates=True)


### Verify data is loaded successfully

In [3]:
display(cointegrated_pairs)

,y,x,hedge_ratio,intercept,adf_stat,p_value,y_is_I1,x_is_I1,is_cointegrated
0,GS.N,MS.N,0.780150,2.350976,-3.666910,0.020199,True,True,True
1,KO.N,PEP.O,0.641464,0.788960,-3.470124,0.035137,True,True,True
2,DAL.N,UAL.O,0.671613,1.067783,-3.460167,0.036094,True,True,True


In [4]:
display(summary_df)

,half_life,hurst,current_zscore,spread_mean,spread_std,adf_stat,adf_pvalue,is_stationary
pair,,,,,,,,
GS.N/MS.N,41.773021,0.402840,0.675478,-5.768157e-16,0.055858,-3.665527,0.004625,True
KO.N/PEP.O,40.323210,0.421018,0.808854,-5.673394e-15,0.046996,-3.468984,0.008820,True
DAL.N/UAL.O,52.863642,0.413517,1.314954,7.682243e-15,0.076182,-3.459317,0.009095,True


In [5]:
display(prices_log)

,NVDA.O,AMD.O,TSM.N,INTC.O,KO.N,PEP.O,COST.O,TGT.N,JPM.N,BAC.N,...,DAL.N,UAL.O,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N,MRK.N,ABBV.N
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,1.606183,2.396075,3.714060,3.846951,3.818591,4.771193,5.189800,4.214052,4.681668,3.397858,...,4.038479,4.233237,4.085144,4.453766,5.200815,3.982677,4.936127,3.542213,3.981684,4.589142
2018-01-03,1.669921,2.446685,3.730741,3.812424,3.816393,4.768564,5.201730,4.207227,4.682687,3.394508,...,4.019801,4.226688,4.097838,4.458409,5.218570,3.999594,4.945634,3.549595,3.980260,4.604670
2018-01-04,1.675179,2.494857,3.725452,3.793915,3.830379,4.773477,5.193934,4.187379,4.691715,3.407511,...,4.019801,4.237868,4.102304,4.467172,5.216728,4.003471,4.945563,3.551772,3.996339,4.598951
2018-01-05,1.683617,2.474856,3.748562,3.800868,3.830162,4.776347,5.186769,4.197954,4.685274,3.412137,...,4.024816,4.239310,4.118338,4.479494,5.230306,4.016644,4.953783,3.553672,3.995287,4.616209
2018-01-08,1.713798,2.507972,3.748091,3.800868,3.828641,4.770600,5.190650,4.207376,4.686750,3.405189,...,4.001498,4.226980,4.132659,4.480514,5.237930,4.020169,4.955052,3.542487,3.989480,4.600057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,3.888345,4.938781,4.636184,3.871201,4.065945,5.122057,6.487177,4.943070,5.120386,3.509454,...,3.716738,3.750680,5.033179,5.925805,5.867572,4.952229,5.046388,3.346389,4.679350,5.043038
2023-12-26,3.897498,4.965708,4.648708,3.921973,4.070052,5.129070,6.491664,4.948973,5.126283,3.522234,...,3.707701,3.739573,5.033114,5.926019,5.871639,4.952441,5.050753,3.346741,4.678699,5.040970
2023-12-27,3.900294,4.984086,4.650621,3.927109,4.072610,5.132263,6.502490,4.958500,5.132263,3.521644,...,3.703522,3.731220,5.032658,5.924443,5.880058,4.944282,5.052097,3.353756,4.681946,5.042651


## Load Out-of-Sample Prices

Retrieve daily closing prices for the cointegrated pair constituents over the out-of-sample window (January 2024 -- December 2025). Only the six constituent tickers are required at this stage; the full 26-ticker universe is re-fetched later for the OOS Engle--Granger re-screen.

In [6]:
tickers_needed = list(set(cointegrated_pairs["y"].tolist() + cointegrated_pairs["x"].tolist()))
print(tickers_needed)

['MS.N', 'UAL.O', 'GS.N', 'KO.N', 'PEP.O', 'DAL.N']


In [7]:
oos_prices_df = get_close_prices(
    tickers=tickers_needed,
    start=TEST_START,
    end=TEST_END,
    interval=INTERVAL
)

len(oos_prices_df)

[cache partial] Fetching 2024-01-01 → 2024-01-01
LSEG session opened.
[cache partial] Skipping (no trading data): LSEG returned no data for ['MS.N', 'UAL.O', 'GS.N', 'KO.N', 'PEP.O', 'DAL.N'] (2024-01-01 to 2024-01-01)


502

In [8]:
oos_prices_df

,KO.N,DAL.N,GS.N,UAL.O,MS.N,PEP.O
Date,,,,,,
2024-01-02,59.82,40.45,388.30,40.72,93.90,172.91
2024-01-03,59.96,38.74,381.79,39.53,91.91,172.95
2024-01-04,59.76,39.20,382.95,40.47,92.15,171.47
2024-01-05,59.67,40.54,386.44,41.76,93.24,168.94
2024-01-08,60.11,41.63,388.86,42.92,93.51,169.11
...,...,...,...,...,...,...
2025-12-24,70.11,70.96,910.78,114.81,181.65,143.74
2025-12-26,69.87,70.85,907.04,114.04,181.87,143.78
2025-12-29,70.16,69.53,892.18,111.45,179.94,144.24


Save raw prices for `04_backtest_results.ipynb`

In [9]:
oos_prices_df.to_csv("../../data/processed/prices_raw_oos.csv")

## Spread Reconstruction with In-Sample Hedge Ratios

Reconstruct the spread $S_t = \ln P^y_t - \hat{\beta}_{\text{IS}}\ln P^x_t - \hat{\alpha}_{\text{IS}}$ using the OLS parameters estimated over the in-sample period. If cointegration is stable, these parameters should produce a stationary spread out-of-sample. Deviations from stationarity indicate structural change.

Convert out-of-sample prices to log prices

In [10]:
oos_prices_log = np.log(oos_prices_df)

Use the estimated hedge ratios and intercept and $\sigma$-bands (mean and standard deviation of spread) from in-sample price series data, to reconstruct the spread for out-of-sample price data

In [11]:
oos_spread = {}

for _, row in cointegrated_pairs.iterrows():
    y_ticker = row["y"]
    x_ticker = row["x"]

    # Ensure both tickers are present in OOS prices
    if y_ticker not in oos_prices_df.columns or x_ticker not in oos_prices_df.columns:
        print(f"Warning: {y_ticker}/{x_ticker} missing from OOS prices, skipping.")
        continue

    spread_oos = compute_spread(
        y           = oos_prices_log[y_ticker],
        x           = oos_prices_log[x_ticker],
        hedge_ratio = row["hedge_ratio"],   # in-sample estimate
        intercept   = row["intercept"],     # in-sample estimate
    )

    oos_spread[f"{y_ticker}/{x_ticker}"] = spread_oos

print(f"Reconstructed {len(oos_spread)} OOS spreads:")
for pair_name, spread in oos_spread.items():
    print(f"  {pair_name}: {spread.first_valid_index().date()} → {spread.last_valid_index().date()}, "
          f"NaNs: {spread.isna().sum()}")


Reconstructed 3 OOS spreads:
  GS.N/MS.N: 2024-01-02 → 2025-12-31, NaNs: 0
  KO.N/PEP.O: 2024-01-02 → 2025-12-31, NaNs: 0
  DAL.N/UAL.O: 2024-01-02 → 2025-12-31, NaNs: 0


Compare in-sample vs out-of-sample spread statistics. An increase in spread standard deviation or mean displacement from zero would indicate regime change; an increase in ADF $p$-value above 0.05 would indicate loss of stationarity.

In [12]:
rows = []
for _, row in cointegrated_pairs.iterrows():
    pair_name = f"{row['y']}/{row['x']}"
    if pair_name not in oos_spread:
        continue

    spread_is  = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    )
    spread_oos = oos_spread[pair_name]

    # is - in sample (2019-2023), oos - out of sample (2024-2025)
    rows.append({
        "pair"          : pair_name,
        "is_mean"       : spread_is.mean(),
        "oos_mean"      : spread_oos.mean(),
        "is_std"        : spread_is.std(),
        "oos_std"       : spread_oos.std(),
        "is_min"        : spread_is.min(),
        "oos_min"       : spread_oos.min(),
        "is_max"        : spread_is.max(),
        "oos_max"       : spread_oos.max(),
    })

sanity_df = pd.DataFrame(rows).set_index("pair")

display(sanity_df)

,is_mean,oos_mean,is_std,oos_std,is_min,oos_min,is_max,oos_max
pair,,,,,,,,
GS.N/MS.N,-5.756385e-16,0.249616,0.055858,0.081732,-0.145132,0.066983,0.110286,0.411057
KO.N/PEP.O,-5.668685e-15,0.169455,0.046996,0.109700,-0.172785,-0.015886,0.111406,0.366825
DAL.N/UAL.O,7.684597e-15,0.026369,0.076182,0.118807,-0.201455,-0.198907,0.174968,0.280153


## Test Whether Cointegration Holds Out-of-Sample

Apply ADF and KPSS tests to the OOS spreads reconstructed under IS parameters. The ADF null is a unit root (non-stationary); the KPSS null is stationarity. A pair fails both tests if: ADF $p > 0.05$ and KPSS $p < 0.05$. All three pairs must meet this joint criterion to reject cointegration persistence.

Run the ADF test on each OOS spread using the IS hedge ratio and intercept. This directly tests whether the IS cointegrating relationship remains valid on OOS data --- the key precondition for OU-based return estimation to be meaningful in the backtest.

In [13]:

oos_results = []
for _, row in cointegrated_pairs.iterrows():
    pair_name = f"{row['y']}/{row['x']}"

    # In-sample summary
    summary_is  = spread_summary(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    )

    # Out-of-sample summary — same hedge ratio and intercept
    summary_oos = spread_summary(
        oos_prices_log[row["y"]], oos_prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    )

    oos_results.append({
        "pair"           : pair_name,
        "adf_pval_is"    : summary_is["adf_pvalue"],
        "adf_pval_oos"   : summary_oos["adf_pvalue"],
        "hl_is"          : summary_is["half_life"],
        "hl_oos"         : summary_oos["half_life"],
        "hurst_is"       : summary_is["hurst"],
        "hurst_oos"      : summary_oos["hurst"],
        "zscore_oos"     : summary_oos["current_zscore"],  # end of OOS period
    })

validation_df = pd.DataFrame(oos_results).set_index("pair")

display(validation_df)

,adf_pval_is,adf_pval_oos,hl_is,hl_oos,hurst_is,hurst_oos,zscore_oos
pair,,,,,,,
GS.N/MS.N,0.004625,0.452421,41.773021,81.073516,0.402840,0.362399,1.015371
KO.N/PEP.O,0.008820,0.621942,40.323210,156.374389,0.421018,0.574742,0.603825
DAL.N/UAL.O,0.009095,0.583524,52.863642,103.107777,0.413517,0.420995,0.872209


The OOS half-life data confirms substantial weakening across all three pairs: GS/MS deteriorated from 41.8 to 81.1 days; KO/PEP nearly quadrupled from 40.3 to 156.4 days; DAL/UAL doubled from 52.9 to 103.1 days. For KO/PEP the Hurst exponent crossed above 0.5 ($0.421 \to 0.575$), signalling a transition from mean-reverting to near-random-walk behaviour. None of the three pairs remains stationary under IS parameters out-of-sample (ADF $p > 0.45$ for all).

### Visualise Results

#### Concatenated Spread with IS/OOS Boundary

In [14]:
n_pairs = len(cointegrated_pairs)
fig_plotly = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)
oos_start = pd.Timestamp("2024-01-01").timestamp() * 1000

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    pair_name   = f"{row['y']}/{row['x']}"
    spread_is   = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos  = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])
    mu    = spread_is.mean()
    sigma = spread_is.std()

    fig_plotly.add_trace(
        go.Scatter(x=spread_full.index, y=spread_full.values,
                   line=dict(width=1), name=pair_name, showlegend=False),
        row=i, col=1
    )

    # IS sigma bands — annotate only on first row, omit kwargs entirely otherwise
    for level, dash, label in [(mu + 2*sigma, "dot",  "+2\u03c3 (IS)"),
                                (mu - 2*sigma, "dot",  "-2\u03c3 (IS)"),
                                (mu,           "dash", "IS Mean")]:
        fig_plotly.add_hline(
            y=level, line=dict(color="gray", dash=dash, width=0.8),
            annotation_text=label,
            annotation_font_size=9,
            annotation_position="right",
            row=i, col=1
        )


    # IS/OOS boundary
    fig_plotly.add_vline(
        x=oos_start, line=dict(color="black", dash="dash", width=1),
        annotation_text="OOS",
        annotation_font_size=9,
        row=i, col=1
    )


fig_plotly.update_yaxes(title_text="Spread")
fig_plotly.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_plotly.update_layout(
    title="Spread Levels — IS vs OOS with In-Sample \u03c3 Bands",
    height=350 * n_pairs, template="plotly_white"
)
fig_plotly.show()

#### save to matplotlib

In [15]:
import scienceplots
import matplotlib.dates as mdates
plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})

n_pairs     = len(cointegrated_pairs)
oos_split   = pd.Timestamp("2024-01-01")
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(6.5, 2.60 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    pair_name   = f"{row['y']}/{row['x']}"
    spread_is   = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos  = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])

    mu    = spread_is.mean()
    sigma = spread_is.std()
    first = (label == panel_labels[0])

    line, = ax.plot(spread_full.index, spread_full.values, linewidth=1,
                    label="Spread" if first else None)

    ax.axhline(mu,             linestyle="--", linewidth=0.8, color="grey",
               label=r"IS mean"          if first else None)
    ax.axhline(mu + 2*sigma,   linestyle=":",  linewidth=0.8, color="grey",
               label=r"IS $\pm2\sigma$"  if first else None)
    ax.axhline(mu - 2*sigma,   linestyle=":",  linewidth=0.8, color="grey")
    ax.axvline(oos_split,      color="black", linestyle="--", linewidth=0.8,
               label="IS/OOS split"      if first else None)
    ax.axhspan(mu - 2*sigma, mu + 2*sigma, alpha=0.06)

    ax.text(0.02, 0.93, f"{label} {pair_name}", transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")
    ax.set_ylim(spread_full.quantile(0.005), spread_full.quantile(0.995))
    ax.set_ylabel("Spread")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02),
           frameon=True, edgecolor="grey")
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.subplots_adjust(hspace=0.08)
fig.savefig("figures/02_spread_is_oos_concat.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

### Spread levels with $\pm 1\sigma / \pm 2\sigma$ bands (OOS only)

In [16]:
fig_oos = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    pair_name  = f"{row['y']}/{row['x']}"
    spread_is  = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos = oos_spread[pair_name].dropna()

    mu    = spread_is.mean()
    sigma = spread_is.std()

    fig_oos.add_trace(
        go.Scatter(x=spread_oos.index, y=spread_oos.values,
                   line=dict(width=1), name=pair_name, showlegend=False),
        row=i, col=1
    )

    for level, dash, label in [(mu + 2*sigma, "dot",  "+2\u03c3 (IS)"),
                                (mu - 2*sigma, "dot",  "-2\u03c3 (IS)"),
                                (mu + sigma,   "dot",  "+1\u03c3 (IS)"),
                                (mu - sigma,   "dot",  "-1\u03c3 (IS)"),
                                (mu,           "dash", "IS Mean")]:
        fig_oos.add_hline(
            y=level, line=dict(color="gray", dash=dash, width=0.8),
            annotation_text=label, annotation_font_size=9,
            annotation_position="right",
            row=i, col=1
        )

fig_oos.update_yaxes(title_text="Spread")
fig_oos.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_oos.update_layout(
    title="Spread Levels — OOS Only with In-Sample \u03c3 Bands",
    height=350 * n_pairs, template="plotly_white"
)
fig_oos.show()


##### Save to Matplotlib

In [17]:
plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(6.5, 2.60 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    pair_name  = f"{row['y']}/{row['x']}"
    spread_is  = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos = oos_spread[pair_name].dropna()

    mu    = spread_is.mean()
    sigma = spread_is.std()
    first = (label == panel_labels[0])

    ax.plot(spread_oos.index, spread_oos.values, linewidth=1,
            label="Spread (OOS)" if first else None)
    ax.axhline(mu,           linestyle="--", linewidth=0.8, color="grey",
               label=r"IS mean" if first else None)
    ax.axhline(mu + 2*sigma, linestyle=":",  linewidth=0.8, color="grey",
               label=r"IS $\pm2\sigma$" if first else None)
    ax.axhline(mu - 2*sigma, linestyle=":",  linewidth=0.8, color="grey")
    ax.axhline(mu + sigma,   linestyle=":",  linewidth=0.6, alpha=0.5, color="grey",
               label=r"IS $\pm1\sigma$" if first else None)
    ax.axhline(mu - sigma,   linestyle=":",  linewidth=0.6, alpha=0.5, color="grey")
    ax.axhspan(mu - 2*sigma, mu + 2*sigma, alpha=0.06)

    ax.set_ylabel("Spread")
    ax.text(0.02, 0.93, f"{label} {pair_name}", transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.02),
           frameon=True, edgecolor="grey")
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.subplots_adjust(hspace=0.08)
fig.savefig("figures/02_spread_oos_levels.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

#### Z-score (In-Sample + Out-of-Sample) with $\pm 2\sigma$ threshold

In [18]:
fig_zscore = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    pair_name  = f"{row['y']}/{row['x']}"
    spread_is  = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])

    zscore_full = compute_zscore(spread_full, window=60)

    fig_zscore.add_trace(
        go.Scatter(x=zscore_full.index, y=zscore_full.values,
                   line=dict(width=1), name=pair_name, showlegend=False),
        row=i, col=1
    )

    for level, dash, label in [(2,  "dot",  "+2\u03c3"),
                                (-2, "dot",  "-2\u03c3"),
                                (0,  "dash", "Mean")]:
        fig_zscore.add_hline(
            y=level, line=dict(color="gray", dash=dash, width=0.8),
            annotation_text=label, annotation_font_size=9,
            annotation_position="right",
            row=i, col=1
        )

    fig_zscore.add_vline(
        x=oos_start, line=dict(color="black", dash="dash", width=1),
        annotation_text="OOS", annotation_font_size=9,
        row=i, col=1
    )

fig_zscore.update_yaxes(title_text="Z-score")
fig_zscore.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_zscore.update_layout(
    title="Z-Score (60-day rolling) — IS vs OOS",
    height=350 * n_pairs, template="plotly_white"
)
fig_zscore.show()


##### Save to Matplotlib

In [19]:
plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(6.5, 2.60 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, (_, row), label in zip(axes, cointegrated_pairs.iterrows(), panel_labels):
    pair_name   = f"{row['y']}/{row['x']}"
    spread_is   = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos  = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])
    zscore_full = compute_zscore(spread_full, window=60)
    first = (label == panel_labels[0])

    ax.plot(zscore_full.index, zscore_full.values, linewidth=1,
            label="Z-score" if first else None)
    ax.axhline(0,  linestyle="--", linewidth=0.8, color="grey",
               label="Mean" if first else None)
    ax.axhline(2,  linestyle=":",  linewidth=0.8, color="grey",
               label=r"$\pm2\sigma$" if first else None)
    ax.axhline(-2, linestyle=":",  linewidth=0.8, color="grey")
    ax.axhline(1,  linestyle=":",  linewidth=0.6, alpha=0.5, color="grey",
               label=r"$\pm1\sigma$" if first else None)
    ax.axhline(-1, linestyle=":",  linewidth=0.6, alpha=0.5, color="grey")
    ax.axvline(oos_split, color="black", linestyle="--", linewidth=0.8,
               label="IS/OOS split" if first else None)
    ax.axhspan(-2, 2, alpha=0.06)

    ax.set_ylabel("Z-score")
    ax.text(0.02, 0.93, f"{label} {pair_name}", transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.02),
           frameon=True, edgecolor="grey")
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.subplots_adjust(hspace=0.08)
fig.savefig("figures/02_zscore_is_oos.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

#### Rolling half-life (full period)

In [20]:
fig_rhl = make_subplots(
    rows=n_pairs, cols=1,
    shared_xaxes=True,
    subplot_titles=[f"{r['y']}/{r['x']}" for _, r in cointegrated_pairs.iterrows()],
    vertical_spacing=0.08
)

for i, (_, row) in enumerate(cointegrated_pairs.iterrows(), start=1):
    pair_name   = f"{row['y']}/{row['x']}"
    spread_is   = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos  = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])

    rhl        = compute_rolling_half_life(spread_full, window=252).replace(np.inf, np.nan)
    rhl_smooth = rhl.ewm(span=21, adjust=False).mean()

    fig_rhl.add_trace(
        go.Scatter(x=rhl.index, y=rhl.values,
                   line=dict(width=0.8, color="steelblue"), opacity=0.3,
                   name="Raw", showlegend=(i == 1)),
        row=i, col=1
    )
    fig_rhl.add_trace(
        go.Scatter(x=rhl_smooth.index, y=rhl_smooth.values,
                   line=dict(width=1.8, color="steelblue"),
                   name="EWM (21d)", showlegend=(i == 1)),
        row=i, col=1
    )
    fig_rhl.add_hline(
        y=60, line=dict(color="red", dash="dash", width=1),
        annotation_text="60d threshold", annotation_font_size=9,
        annotation_position="top right",
        row=i, col=1
    )
    fig_rhl.add_vline(
        x=oos_start, line=dict(color="black", dash="dash", width=1),
        annotation_text="OOS", annotation_font_size=9,
        row=i, col=1
    )

fig_rhl.update_yaxes(title_text="Half-Life (days)", range=[0, 500])
fig_rhl.update_xaxes(title_text="Date", row=n_pairs, col=1)
fig_rhl.update_layout(
    title="Rolling Half-Life (252-day window, EWM-smoothed) — IS vs OOS",
    height=350 * n_pairs, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_rhl.show()


##### Save to Matplotlib

In [21]:
plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})

prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(6.5, 2.60 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, (_, row), label, color in zip(axes, cointegrated_pairs.iterrows(), panel_labels, prop_cycle):
    pair_name   = f"{row['y']}/{row['x']}"
    spread_is   = compute_spread(
        prices_log[row["y"]], prices_log[row["x"]],
        row["hedge_ratio"], row["intercept"]
    ).dropna()
    spread_oos  = oos_spread[pair_name].dropna()
    spread_full = pd.concat([spread_is, spread_oos])
    rhl         = compute_rolling_half_life(spread_full, window=252).replace(np.inf, np.nan)
    rhl_smooth  = rhl.ewm(span=21, adjust=False).mean()
    first       = (label == panel_labels[0])

    ax.plot(rhl.index, rhl.values,
            linewidth=0.8, alpha=0.3, color=color,
            label="Raw" if first else None)
    ax.plot(rhl_smooth.index, rhl_smooth.values,
            linewidth=1.8, color=color,
            label="EWM (21d)" if first else None)
    ax.axhline(60, linestyle="--", linewidth=1, color="black",
               label="60d threshold" if first else None)
    ax.fill_between(rhl_smooth.index, rhl_smooth.values, 60,
                    where=(rhl_smooth.fillna(0).values > 60),
                    alpha=0.15, color=color)
    ax.axvline(oos_split, color="black", linestyle="--", linewidth=0.8,
               label="IS/OOS split" if first else None)

    ax.set_ylabel(r"Half-Life (days)")
    ax.set_ylim(0, 500)
    ax.text(0.02, 0.93, f"{label} {pair_name}", transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02),
           frameon=True, edgecolor="grey")
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.subplots_adjust(hspace=0.08)
fig.savefig("figures/02_rolling_halflife_is_oos.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

## Out-of-Sample Engle--Granger Cointegration Re-Test

Two questions are addressed: (1) do pairs cointegrated in-sample remain cointegrated out-of-sample when parameters are re-estimated on OOS data? (2) do any pairs not cointegrated in-sample become cointegrated out-of-sample? A pair is classified as *Persistent* if cointegrated in both periods, *Degraded* if only in-sample, *New* if only out-of-sample, and *Never* if neither.

### Fetch OOS Prices for All Tickers

The earlier OOS price fetch only covered IS-cointegrated tickers. Here we fetch all tickers from `TICKERS` so we can run Engle-Granger on all `CANDIDATE_PAIRS`.

In [22]:
# Fetch OOS prices for all tickers (not just IS-cointegrated ones)
all_oos_prices_df = get_close_prices(
    tickers=TICKERS,
    start=TEST_START,
    end=TEST_END,
    interval=INTERVAL
)

all_oos_prices_log = np.log(all_oos_prices_df)

print(f"OOS price data: {all_oos_prices_df.shape[0]} trading days × {all_oos_prices_df.shape[1]} tickers")
all_oos_prices_df.head()

[cache hit] ABBV-N_AMD-O_AMZN-O_BAC-N_COST-O_CVX-N_D__1D__e7b053e145.csv
OOS price data: 502 trading days × 26 tickers


,NVDA.O,AMD.O,TSM.N,INTC.O,KO.N,PEP.O,COST.O,TGT.N,JPM.N,BAC.N,...,DAL.N,UAL.O,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N,MRK.N,ABBV.N
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,48.168,138.58,101.53,47.80,59.82,172.91,650.65,143.10,172.08,33.90,...,40.45,40.72,149.93,370.87,346.29,138.17,159.97,29.73,113.24,159.82
2024-01-03,47.569,135.32,100.17,47.05,59.96,172.95,644.69,138.67,171.33,33.53,...,38.74,39.53,148.47,370.60,344.47,138.92,160.97,29.73,114.77,160.46
2024-01-04,47.998,136.01,99.13,46.87,59.76,171.47,648.35,140.25,171.41,33.80,...,39.20,40.47,144.57,367.94,347.12,136.39,160.63,29.09,117.01,161.46
2024-01-05,49.097,138.58,99.61,46.89,59.67,168.94,656.01,140.75,172.27,34.43,...,40.54,41.76,145.24,367.75,351.95,135.73,161.13,29.47,117.22,162.14
2024-01-08,52.253,146.18,102.24,48.45,60.11,169.11,661.69,141.73,172.02,34.16,...,41.63,42.92,149.10,374.69,358.66,138.84,161.53,29.58,117.38,161.43


### Re-run Engle-Granger on All Candidate Pairs (OOS)

Run `screen_pairs` on out-of-sample log prices across all `CANDIDATE_PAIRS` — identical to the in-sample screening in `01_spread_exploration.ipynb`.

In [23]:
# Run Engle-Granger on all candidate pairs using OOS log prices
oos_eg_results = screen_pairs(all_oos_prices_log, CANDIDATE_PAIRS, significance=0.05)
oos_eg_results

,y,x,hedge_ratio,intercept,adf_stat,p_value,y_is_I1,x_is_I1,is_cointegrated
0,NVDA.O,TSM.N,1.119949,-1.038056,-3.845872,0.011762,False,True,False
1,GS.N,MS.N,1.120199,0.975161,-3.683606,0.019234,True,True,True
2,NVDA.O,AMD.O,0.292898,3.360723,-3.265316,0.059667,False,True,False
3,INTC.O,AMD.O,0.830877,-0.854504,-3.163016,0.076334,True,True,False
4,CVX.N,XOM.N,0.470044,2.804848,-2.968660,0.117844,False,False,False
5,COST.O,TGT.N,-0.318185,8.317926,-2.811585,0.162026,True,True,False
6,PFE.N,JNJ.N,-0.165717,4.112660,-2.750785,0.181847,True,True,False
7,AMZN.O,GOOGL.O,0.440014,3.004628,-2.742354,0.184717,True,True,False
8,KO.N,PEP.O,-0.349815,5.961571,-2.635529,0.223618,True,True,False
9,SLB.N,HAL.N,0.676051,1.459406,-2.586824,0.242175,True,True,False


### IS vs OOS Full Comparison Table

Merge in-sample (`01_spread_exploration`) results with out-of-sample Engle-Granger results across all candidate pairs. Pairs not tested in-sample (excluded from `cointegrated_pairs`) are marked `is_cointegrated = False`.

In [24]:
# Build a lookup for IS results (all candidate pairs from 01_spread_exploration)
# Load the full IS screening results — reload prices_log which has all tickers
is_prices_log = pd.read_csv("../../data/processed/prices_log.csv", index_col=0, parse_dates=True)
is_all_results = screen_pairs(is_prices_log, CANDIDATE_PAIRS, significance=0.05)

comparison_rows = []

for _, oos_row in oos_eg_results.iterrows():
    y_t, x_t = oos_row["y"], oos_row["x"]
    pair_name = f"{y_t}/{x_t}"

    # Match IS result by ticker set (ordering may differ)
    mask = (
        ((is_all_results["y"] == y_t) & (is_all_results["x"] == x_t)) |
        ((is_all_results["y"] == x_t) & (is_all_results["x"] == y_t))
    )
    is_row = is_all_results[mask]

    if is_row.empty:
        continue

    is_r = is_row.iloc[0]
    comparison_rows.append({
        "pair"              : pair_name,
        "is_p_value"        : round(float(is_r["p_value"]), 4),
        "oos_p_value"       : round(float(oos_row["p_value"]), 4),
        "is_hedge_ratio"    : round(float(is_r["hedge_ratio"]), 4),
        "oos_hedge_ratio"   : round(float(oos_row["hedge_ratio"]), 4),
        "hedge_ratio_drift" : round(abs(float(oos_row["hedge_ratio"]) - float(is_r["hedge_ratio"])), 4),
        "is_cointegrated"   : bool(is_r["is_cointegrated"]),
        "oos_cointegrated"  : bool(oos_row["is_cointegrated"]),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("pair")
display(comparison_df.sort_values(by="oos_p_value"))

,is_p_value,oos_p_value,is_hedge_ratio,oos_hedge_ratio,hedge_ratio_drift,is_cointegrated,oos_cointegrated
pair,,,,,,,
NVDA.O/TSM.N,0.8172,0.0118,0.4763,1.1199,0.6437,False,False
GS.N/MS.N,0.0202,0.0192,0.7802,1.1202,0.3400,True,True
NVDA.O/AMD.O,0.3684,0.0597,0.8111,0.2929,0.5182,False,False
INTC.O/AMD.O,0.5501,0.0763,-0.0810,0.8309,0.9118,False,False
CVX.N/XOM.N,0.0644,0.1178,0.6981,0.4700,0.2280,False,False
COST.O/TGT.N,0.9251,0.1620,0.8562,-0.3182,1.1744,False,False
PFE.N/JNJ.N,0.5761,0.1818,0.4228,-0.1657,0.5885,False,False
AMZN.O/GOOGL.O,0.3053,0.1847,0.6303,0.4400,0.1903,False,False
KO.N/PEP.O,0.0351,0.2236,0.6415,-0.3498,0.9913,True,False


### Visualise IS vs OOS Engle-Granger p-values

Grouped bar chart across all candidate pairs. Hatching indicates the pair fails the 5% significance threshold. This directly shows which relationships are persistent, which degrade, and whether any new cointegrated pairs emerge out-of-sample.

In [25]:
import scienceplots
import matplotlib.patches as mpatches

plt.style.use(["science", "high-contrast", "ieee"])
plt.rcParams["text.usetex"] = True
plt.rcParams.update({
    'font.size':        10,
    'axes.labelsize':   10,
    'axes.titlesize':   10,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'legend.fontsize':  9,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],  # LaTeX default font
})

pairs         = comparison_df.index.tolist()
n_pairs       = len(pairs)
panel_letters = [f"({chr(97 + i)})" for i in range(n_pairs)]
x             = np.arange(n_pairs)
width = 0.35

fig, ax = plt.subplots(figsize=(6.5, 3.20))

bars_is  = ax.bar(x - width / 2, comparison_df["is_p_value"],  width, label="In-Sample (2018--2023)")
bars_oos = ax.bar(x + width / 2, comparison_df["oos_p_value"], width, label="Out-of-Sample (2024--2025)")

# Hatch bars that fail the 5% threshold
for bar, p in zip(bars_is, comparison_df["is_p_value"]):
    if p >= 0.05:
        bar.set_hatch("///")
for bar, p in zip(bars_oos, comparison_df["oos_p_value"]):
    if p >= 0.05:
        bar.set_hatch("///")

ax.axhline(0.05, linestyle="--", linewidth=1, color="black")
ax.text(
    0.92, 0.055, r"$5\%$ threshold",
    transform=ax.get_yaxis_transform(),
    ha="right", va="bottom", fontsize=9,
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=2)
)

ax.set_xticks(x)
tick_labels = [f"{letter}" for letter, pair in zip(panel_letters, pairs)]
ax.set_xticklabels(tick_labels, rotation=0, fontsize=9)
ax.set_ylabel("Engle-Granger p-value")
ax.set_ylim(0, max(comparison_df[["is_p_value", "oos_p_value"]].max().max() * 1.45, 0.14))

ax.set_xlabel("")

pass_patch = mpatches.Patch(facecolor="white", edgecolor="black", linewidth=0.8,
                            label=r"Cointegrated (p $<$ 0.05)")
fail_patch = mpatches.Patch(facecolor="white", edgecolor="black", linewidth=0.8, hatch="///",
                            label=r"Not cointegrated (p $\geq$ 0.05)")
handles, labels_ = ax.get_legend_handles_labels()
fig.legend(handles + [pass_patch, fail_patch],
           labels_ + [pass_patch.get_label(), fail_patch.get_label()],
           loc="lower center", ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.16),
           frameon=True, edgecolor="grey")

fig.tight_layout(rect=[0, 0.1, 1, 1])
fig.savefig("figures/02_oos_eg_pvalues.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

Summary of persistence of cointegration across In Sample vs Out of Sample

In [26]:
# Summary: persistence of cointegration across IS → OOS
persistent   = comparison_df[ comparison_df["is_cointegrated"] &  comparison_df["oos_cointegrated"]]
degraded     = comparison_df[ comparison_df["is_cointegrated"] & ~comparison_df["oos_cointegrated"]]
new_oos      = comparison_df[~comparison_df["is_cointegrated"] &  comparison_df["oos_cointegrated"]]
never_coint  = comparison_df[~comparison_df["is_cointegrated"] & ~comparison_df["oos_cointegrated"]]

print("=== OOS Engle-Granger Cointegration Summary ===\n")
print(f"Persistent (cointegrated IS + OOS):  {len(persistent)} pair(s)")
for p in persistent.index: print(f"   + {p}")

print(f"\nDegraded  (cointegrated IS only):    {len(degraded)} pair(s)")
for p in degraded.index:   print(f"   - {p}")

print(f"\nNew OOS   (cointegrated OOS only):   {len(new_oos)} pair(s)")
for p in new_oos.index:    print(f"   ~ {p}")

print(f"\nNever cointegrated:                  {len(never_coint)} pair(s)")
for p in never_coint.index: print(f"   x {p}")

=== OOS Engle-Granger Cointegration Summary ===

Persistent (cointegrated IS + OOS):  1 pair(s)
   + GS.N/MS.N

Degraded  (cointegrated IS only):    2 pair(s)
   - KO.N/PEP.O
   - DAL.N/UAL.O

New OOS   (cointegrated OOS only):   0 pair(s)

Never cointegrated:                  13 pair(s)
   x NVDA.O/TSM.N
   x NVDA.O/AMD.O
   x INTC.O/AMD.O
   x CVX.N/XOM.N
   x COST.O/TGT.N
   x PFE.N/JNJ.N
   x AMZN.O/GOOGL.O
   x SLB.N/HAL.N
   x AMZN.O/MSFT.O
   x META.O/GOOGL.O
   x BAC.N/JPM.N
   x ABBV.N/MRK.N
   x INTC.O/TSM.N


## Statistical Tests: KPSS Complement and Chow Structural Break

Two formal tests are applied to sharpen the cointegration breakdown diagnosis. The **KPSS test** complements the ADF test: ADF rejects the null of a unit root, while KPSS tests the null of stationarity. Disagreement between the two provides a more nuanced picture than either test alone. The **Chow $F$-test** directly tests whether the OLS hedge ratio and intercept are stable at the IS/OOS boundary.

In [27]:
from statsmodels.tsa.stattools import kpss

def kpss_test(series, regression='c', nlags='auto'):
    """KPSS test. H0 = series is stationary. Reject H0 => non-stationary."""
    clean = series.dropna()
    stat, p_value, n_lags, crit = kpss(clean, regression=regression, nlags=nlags)
    return {'kpss_stat': stat, 'p_value': p_value, 'n_lags': n_lags,
            'critical_values': crit, 'is_stationary': p_value > 0.05}

print('KPSS test on IS and OOS spreads (H0 = stationary, reject => non-stationary)\n')
kpss_rows = []
for _, row in cointegrated_pairs.iterrows():
    pair_name = f"{row['y']}/{row['x']}"
    spread_is  = compute_spread(prices_log[row['y']], prices_log[row['x']],
                                row['hedge_ratio'], row['intercept'])
    spread_oos_ = compute_spread(oos_prices_log[row['y']], oos_prices_log[row['x']],
                                 row['hedge_ratio'], row['intercept'])

    adf_is  = adf_test(spread_is)
    adf_oos = adf_test(spread_oos_)
    k_is    = kpss_test(spread_is)
    k_oos   = kpss_test(spread_oos_)

    # Consensus: mean-reverting if ADF rejects non-stationarity AND KPSS fails to reject stationarity
    consensus_is  = (not adf_is['is_stationary'] is False) and adf_is['is_stationary'] and k_is['is_stationary']
    consensus_oos = adf_oos['is_stationary'] and k_oos['is_stationary']

    kpss_rows.append({
        'Pair':            pair_name,
        'ADF p (IS)':      round(adf_is['p_value'], 4),
        'KPSS p (IS)':     round(k_is['p_value'], 4),
        'MR Consensus IS': 'Yes' if consensus_is else 'No',
        'ADF p (OOS)':     round(adf_oos['p_value'], 4),
        'KPSS p (OOS)':    round(k_oos['p_value'], 4),
        'MR Consensus OOS':'Yes' if consensus_oos else 'No',
    })

kpss_df = pd.DataFrame(kpss_rows).set_index('Pair')
display(kpss_df)
print('\nInterpretation: ADF p < 0.05 AND KPSS p > 0.05 => mean-reverting (stationary spread).')

KPSS test on IS and OOS spreads (H0 = stationary, reject => non-stationary)



/var/folders/w0/fm4jsyzj2rv6f2269jdg1h7r0000gn/T/ipykernel_41675/1230972735.py:6: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.


/var/folders/w0/fm4jsyzj2rv6f2269jdg1h7r0000gn/T/ipykernel_41675/1230972735.py:6: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


/var/folders/w0/fm4jsyzj2rv6f2269jdg1h7r0000gn/T/ipykernel_41675/1230972735.py:6: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.


/var/folders/w0/fm4jsyzj2rv6f2269jdg1h7r0000gn/T/ipykernel_41675/1230972735.py:6: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned

,ADF p (IS),KPSS p (IS),MR Consensus IS,ADF p (OOS),KPSS p (OOS),MR Consensus OOS
Pair,,,,,,
GS.N/MS.N,0.0046,0.0188,No,0.4524,0.01,No
KO.N/PEP.O,0.0088,0.1000,Yes,0.6219,0.01,No
DAL.N/UAL.O,0.0091,0.0803,Yes,0.5835,0.01,No



Interpretation: ADF p < 0.05 AND KPSS p > 0.05 => mean-reverting (stationary spread).


In [28]:
import statsmodels.api as sm
from scipy import stats

def chow_test(y_is, x_is, y_oos, x_oos):
    """
    Chow test for structural break in cointegrating regression y = a + b*x.
    Null: coefficients are equal across IS and OOS samples.
    F-statistic follows F(k, n1+n2-2k) under H0.
    """
    def _ols_rss(y, x):
        X = sm.add_constant(x.values)
        res = sm.OLS(y.values, X).fit()
        return res.ssr, len(y), res.df_resid

    rss_is,  n1, _ = _ols_rss(y_is,  x_is)
    rss_oos, n2, _ = _ols_rss(y_oos, x_oos)

    # Pooled regression
    y_pool = pd.concat([y_is.reset_index(drop=True), y_oos.reset_index(drop=True)])
    x_pool = pd.concat([x_is.reset_index(drop=True), x_oos.reset_index(drop=True)])
    rss_pool, _, _ = _ols_rss(y_pool, x_pool)

    k  = 2  # intercept + slope
    n  = n1 + n2
    F  = ((rss_pool - (rss_is + rss_oos)) / k) / ((rss_is + rss_oos) / (n - 2 * k))
    p  = 1 - stats.f.cdf(F, k, n - 2 * k)
    return {'F_stat': F, 'p_value': p, 'df1': k, 'df2': n - 2 * k,
            'structural_break': p < 0.05}

print('Chow Test for Structural Break (H0 = no structural break across IS/OOS split)\n')
chow_rows = []
for _, row in cointegrated_pairs.iterrows():
    pair_name = f"{row['y']}/{row['x']}"
    result = chow_test(
        prices_log[row['y']],  prices_log[row['x']],
        oos_prices_log[row['y']], oos_prices_log[row['x']]
    )
    chow_rows.append({
        'Pair':             pair_name,
        'F-statistic':      round(result['F_stat'], 4),
        'p-value':          round(result['p_value'], 4),
        'df1 / df2':        f"{result['df1']} / {result['df2']}",
        'Structural Break': 'Yes (p<0.05)' if result['structural_break'] else 'No',
    })
    print(f"{pair_name}: F={result['F_stat']:.4f}, p={result['p_value']:.4f} "
          f"=> {'BREAK DETECTED' if result['structural_break'] else 'stable'}")

chow_df = pd.DataFrame(chow_rows).set_index('Pair')
display(chow_df)
print('\nInterpretation: p < 0.05 => reject null of stable coefficients => structural break detected.')

Chow Test for Structural Break (H0 = no structural break across IS/OOS split)

GS.N/MS.N: F=2406.2553, p=0.0000 => BREAK DETECTED
KO.N/PEP.O: F=3078.3558, p=0.0000 => BREAK DETECTED
DAL.N/UAL.O: F=269.5353, p=0.0000 => BREAK DETECTED


,F-statistic,p-value,df1 / df2,Structural Break
Pair,,,,
GS.N/MS.N,2406.2553,0.0,2 / 2007,Yes (p<0.05)
KO.N/PEP.O,3078.3558,0.0,2 / 2007,Yes (p<0.05)
DAL.N/UAL.O,269.5353,0.0,2 / 2007,Yes (p<0.05)



Interpretation: p < 0.05 => reject null of stable coefficients => structural break detected.


**Out-of-Sample Engle--Granger summary.** Re-running on OOS data with re-estimated parameters, only GS/MS retains a significant cointegration signal ($p = 0.019$). KO/PEP ($p = 0.224$) and DAL/UAL ($p = 0.621$) both degrade. No new pairs emerge. The Chow test confirms universal structural breaks (GS/MS: $F = 2406$, KO/PEP: $F = 3078$, DAL/UAL: $F = 270$; all $p < 0.001$), consistent with the large hedge ratio drifts observed in the IS vs OOS comparison table.

## Save DataFrames to CSV

In [61]:
# Save OOS price logs / dataframe for validation and sanity checks

oos_prices_log.to_csv("../../data/processed/prices_log_oos.csv")
validation_df.to_csv("../../data/processed/validation_df.csv")
sanity_df.to_csv("../../data/processed/sanity_df.csv")


## Conclusion

Out-of-sample validation (January 2024 -- December 2025) reveals that the cointegrating relationships identified in-sample do not persist under IS parameters. The key results are summarised below:

| Pair | IS $p$-value | OOS $p$-value | IS cointegrated | OOS cointegrated | Outcome |
|------|-------------|--------------|-----------------|------------------|---------|
| GS.N / MS.N | 0.020 | 0.019 | Yes | Yes | **Persistent** |
| KO.N / PEP.O | 0.035 | 0.224 | Yes | No | **Degraded** |
| DAL.N / UAL.O | 0.036 | 0.621 | Yes | No | **Degraded** |

**Structural break.** Chow tests detect statistically significant parameter shifts for all three pairs at the IS/OOS boundary ($F > 269$, $p < 0.001$ in all cases), confirming that the in-sample hedge ratios are invalid for OOS spread construction. The KPSS test independently rejects stationarity of all three OOS spreads ($p \leq 0.010$).

**Implication for the backtest.** The OU-implied return estimates in `03_return_estimation.ipynb` are computed from IS spread parameters and applied to OOS data in `04_backtest_results.ipynb`. The cointegration breakdown documented here quantifies the structural risk under which those estimates operate, and motivates the statistical significance tests in notebook 04.